# Weather ETL Pipeline

This project extracts real-time weather data from the OpenWeather API,
transforms the data using Pandas, and stores the cleaned dataset as a CSV file.

In [2]:
%pip install requests pandas

In [18]:
import requests
import pandas as pd
from datetime import datetime

In [19]:
API_KEY = "a5b858ace0f9224731b482fb26e5c5c8"

url = "https://api.openweathermap.org/data/2.5/weather"

In [20]:
cities = ["Lagos", "Ibadan", "Abuja"]

In [22]:
weather_records = []

In [31]:
for city in cities:

    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    record = {
        "City": data["name"],
        "Temperature_C": data["main"]["temp"],
        "Humidity_Pct": data["main"]["humidity"],
        "Weather_Condition": data["weather"][0]["main"],
        "Wind_Speed_mps": data["wind"]["speed"],
        "Date_Time": datetime.fromtimestamp(data["dt"])
    }

    weather_records.append(record)

In [32]:
weather_records

[{'City': 'Lagos',
  'Temperature_C': 27.78,
  'Humidity_Pct': 71,
  'Weather_Condition': 'Clouds',
  'Wind_Speed_mps': 3.57,
  'Date_Time': datetime.datetime(2026, 8, 17, 16, 22, 14)},
 {'City': 'Ibadan',
  'Temperature_C': 26.68,
  'Humidity_Pct': 80,
  'Weather_Condition': 'Rain',
  'Wind_Speed_mps': 2.13,
  'Date_Time': datetime.datetime(2026, 8, 17, 16, 22, 29)},
 {'City': 'Abuja',
  'Temperature_C': 28.08,
  'Humidity_Pct': 79,
  'Weather_Condition': 'Rain',
  'Wind_Speed_mps': 1.07,
  'Date_Time': datetime.datetime(2026, 8, 17, 16, 26, 3)}]

In [38]:
weather_df = pd.DataFrame(weather_records)

weather_df

,City,Temperature_C,Humidity_Pct,Weather_Condition,Wind_Speed_mps,Date_Time
0,Lagos,27.78,71,Clouds,3.57,2026-08-17 16:22:14
1,Ibadan,26.68,80,Rain,2.13,2026-08-17 16:22:29
2,Abuja,28.08,79,Rain,1.07,2026-08-17 16:26:03


In [34]:
weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   City               3 non-null      object        
 1   Temperature_C      3 non-null      float64       
 2   Humidity_Pct       3 non-null      int64         
 3   Weather_Condition  3 non-null      object        
 4   Wind_Speed_mps     3 non-null      float64       
 5   Date_Time          3 non-null      datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(1), object(2)
memory usage: 276.0+ bytes


In [35]:
weather_df.isnull().sum()

,0
City,0
Temperature_C,0
Humidity_Pct,0
Weather_Condition,0
Wind_Speed_mps,0
Date_Time,0


In [36]:
weather_df.duplicated().sum()

np.int64(0)

In [40]:
weather_df["Date"] = weather_df["Date_Time"].dt.date
weather_df["Time"] = weather_df["Date_Time"].dt.time

In [42]:
weather_df = weather_df.sort_values(
    by="Temperature_C",
    ascending=False
)

In [44]:
weather_df = weather_df.reset_index(drop=True)

In [45]:
weather_df

,City,Temperature_C,Humidity_Pct,Weather_Condition,Wind_Speed_mps,Date_Time,Date,Time
0,Abuja,28.08,79,Rain,1.07,2026-08-17 16:26:03,2026-08-17,16:26:03
1,Lagos,27.78,71,Clouds,3.57,2026-08-17 16:22:14,2026-08-17,16:22:14
2,Ibadan,26.68,80,Rain,2.13,2026-08-17 16:22:29,2026-08-17,16:22:29


In [46]:
weather_df.to_csv("weather_data.csv", index=False)

In [47]:
saved_weather = pd.read_csv("weather_data.csv")

saved_weather

,City,Temperature_C,Humidity_Pct,Weather_Condition,Wind_Speed_mps,Date_Time,Date,Time
0,Abuja,28.08,79,Rain,1.07,2026-08-17 16:26:03,2026-08-17,16:26:03
1,Lagos,27.78,71,Clouds,3.57,2026-08-17 16:22:14,2026-08-17,16:22:14
2,Ibadan,26.68,80,Rain,2.13,2026-08-17 16:22:29,2026-08-17,16:22:29


BASIC ANALYSIS

In [48]:
#Find the Hottest City(City with the highest temperature)

hottest_city = weather_df.loc[
    weather_df["Temperature_C"].idxmax(),
    ["City", "Temperature_C"]
]

hottest_city

,0
City,Abuja
Temperature_C,28.08


In [50]:
#Find the Coldest City(City with the lowest temperature)

coldest_city = weather_df.loc[
    weather_df["Temperature_C"].idxmin(),
    ["City", "Temperature_C"]
]

coldest_city

,2
City,Ibadan
Temperature_C,26.68


In [52]:
#Find the City with the average temperature

average_temperature = weather_df["Temperature_C"].mean()

average_temperature

np.float64(27.513333333333332)

In [53]:
highest_humidity = weather_df.loc[
    weather_df["Humidity_Pct"].idxmax(),
    ["City", "Humidity_Pct"]
]

highest_humidity

,2
City,Ibadan
Humidity_Pct,80


In [55]:
#The frequency of the weather condition

weather_df[["City", "Weather_Condition"]]

weather_df["Weather_Condition"].value_counts()

,count
Weather_Condition,
Rain,2
Clouds,1


In [57]:
#Highest Wind Speed City

highest_wind = weather_df.loc[
    weather_df["Wind_Speed_mps"].idxmax(),
    ["City", "Wind_Speed_mps"]
]

highest_wind

,1
City,Lagos
Wind_Speed_mps,3.57


In [58]:
#Descriptive Statistics

weather_df.describe()

,Temperature_C,Humidity_Pct,Wind_Speed_mps,Date_Time
count,3.000000,3.000000,3.000000,3
mean,27.513333,76.666667,2.256667,2026-08-17 16:23:35.333333248
min,26.680000,71.000000,1.070000,2026-08-17 16:22:14
25%,27.230000,75.000000,1.600000,2026-08-17 16:22:21.500000
50%,27.780000,79.000000,2.130000,2026-08-17 16:22:29
75%,27.930000,79.500000,2.850000,2026-08-17 16:24:16
max,28.080000,80.000000,3.570000,2026-08-17 16:26:03
std,0.737111,4.932883,1.254804,NaN


In [ ]:
# Summary Findings.
# Abuja recorded the highest temperature at 28.08°C.
# Ibadan recorded the lowest temperature at 26.68°C.
# The average temperature across the three cities was approximately 27.51°C.
# Ibadan recorded the highest humidity at 80%.
# Ibadan and Abuja reported rain, while Lagos reported cloudy conditions.
# Lagos recorded the highest wind speed at 3.57 m/s.